<a href="https://colab.research.google.com/github/yukirochi/Credit-Card-Fraud-Detection/blob/main/Credit_Card_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the specific file in the dataset you want to load
file_path = "fraudTest.csv"

# Use dataset_load instead of load_dataset
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "tusharbhadouria/credit-card-fraud-detection",
  file_path,
)

print("First 5 records:\n", df.head())

100%|██████████| 143M/143M [00:03<00:00, 47.5MB/s]


First 5 records:
    Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2020-06-21 12:14:25  2291163933867244   
1           1   2020-06-21 12:14:33  3573030041201292   
2           2   2020-06-21 12:14:53  3598215285024754   
3           3   2020-06-21 12:15:15  3591919803438423   
4           4   2020-06-21 12:15:17  3526826139003047   

                               merchant        category    amt   first  \
0                 fraud_Kirlin and Sons   personal_care   2.86    Jeff   
1                  fraud_Sporer-Keebler   personal_care  29.84  Joanne   
2  fraud_Swaniawski, Nitzsche and Welch  health_fitness  41.28  Ashley   
3                     fraud_Haley Group        misc_pos  60.05   Brian   
4                 fraud_Johnston-Casper          travel   3.19  Nathan   

       last gender                       street  ...      lat      long  \
0   Elliott      M            351 Darlene Green  ...  33.9659  -80.9355   
1  Williams      F             3638 Marsh Un

In [2]:


df = df.dropna().drop(columns=['first','last','gender','dob','street','trans_date_trans_time','job','cc_num','trans_num'])
print(f'REMAINING COLUMN \n{df.columns}')

print(f' CHECK FOR NULL VALUES \n{df.isna().sum()}')

print(f' COUNT OF EACH UNIQUE MERCHANT {df['merchant'].value_counts()}')

print(len(df))

REMAINING COLUMN 
Index(['Unnamed: 0', 'merchant', 'category', 'amt', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long',
       'is_fraud'],
      dtype='object')
 CHECK FOR NULL VALUES 
Unnamed: 0    0
merchant      0
category      0
amt           0
city          0
state         0
zip           0
lat           0
long          0
city_pop      0
unix_time     0
merch_lat     0
merch_long    0
is_fraud      0
dtype: int64
 COUNT OF EACH UNIQUE MERCHANT merchant
fraud_Kilback LLC                        1859
fraud_Cormier LLC                        1597
fraud_Schumm PLC                         1561
fraud_Kuhn LLC                           1521
fraud_Dickinson Ltd                      1519
                                         ... 
fraud_Treutel-King                        323
fraud_Satterfield-Lowe                    319
fraud_Kessler Group                       318
fraud_Jerde-Hermann                       312
fraud_Ritchie, Bradtke and St

In [16]:
from sklearn.preprocessing import LabelEncoder


label = LabelEncoder()

df['merchant'] = label.fit_transform(df['merchant'])
df['state'] = label.fit_transform(df['state'])
df['category'] = label.fit_transform(df['category'])
df['city'] = label.fit_transform(df['city'])



print(df.iloc[1,:])
print(df.head())
print(df.corr()['is_fraud'].sort_values(ascending=False))

Unnamed: 0    1.000000e+00
merchant      5.910000e+02
category      1.000000e+01
amt           2.984000e+01
city          1.600000e+01
state         4.300000e+01
zip           8.400200e+04
lat           4.032070e+01
long         -1.104360e+02
city_pop      3.020000e+02
unix_time     1.371817e+09
merch_lat     3.945050e+01
merch_long   -1.099604e+02
is_fraud      0.000000e+00
Name: 1, dtype: float64
   Unnamed: 0  merchant  category    amt  city  state    zip      lat  \
0           0       319        10   2.86   157     39  29209  33.9659   
1           1       591        10  29.84    16     43  84002  40.3207   
2           2       611         5  41.28    61     33  11710  40.6729   
3           3       222         9  60.05   764      8  32780  28.5697   
4           4       292        13   3.19   247     21  49632  44.2529   

       long  city_pop   unix_time  merch_lat  merch_long  is_fraud  
0  -80.9355    333497  1371816865  33.986391  -81.200714         0  
1 -110.4360       302

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



x = df.drop(columns=['is_fraud'])
y = df['is_fraud']

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state= 42)
print(x.columns)


scalar = StandardScaler()
scalar_val_cols = ['amt', 'city_pop', 'lat', 'long']
x_train[scalar_val_cols] = scalar.fit_transform(x_train[scalar_val_cols])

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(x_train, y_train)

y_pred =  rf_model.predict(x_test)

print(classification_report(y_test, y_pred))



Index(['Unnamed: 0', 'merchant', 'category', 'amt', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long'],
      dtype='object')
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    110718
           1       0.00      0.00      0.00       426

    accuracy                           1.00    111144
   macro avg       0.50      0.50      0.50    111144
weighted avg       0.99      1.00      0.99    111144



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
import xgboost as xgb
from sklearn.metrics import classification_report


ratio = float(sum(y_train == 0)) / sum(y_train == 1)
print(f"Scale Pos Weight (Normal/Fraud Ratio): {ratio:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=100,           # Number of sequential trees
    scale_pos_weight=ratio,     # Forces model to focus on the rare fraud cases
    learning_rate=0.1,          # Step size shrinkage to prevent overfitting
    random_state=42,
    eval_metric='logloss'       # Suppresses standard warning messages
)

xgb_model.fit(x_train, y_train)

y_pred = xgb_model.predict(x_test)

probabilities = xgb_model.predict_proba(x_test)[:, 1] # Gets probability of Class 1 (Fraud)

# Lower the threshold: Predict fraud if it's even 15% sure
custom_threshold = 0.15
y_pred_custom = (probabilities >= custom_threshold).astype(int)

# Check the new results
print(classification_report(y_test, y_pred_custom))

Scale Pos Weight (Normal/Fraud Ratio): 257.62
              precision    recall  f1-score   support

           0       1.00      0.55      0.71    110718
           1       0.01      0.69      0.01       426

    accuracy                           0.55    111144
   macro avg       0.50      0.62      0.36    111144
weighted avg       0.99      0.55      0.71    111144

